# Entrenamiento BiLSTM-CRF para extraer datos personales
**Antes de empezar:** menú *Entorno de ejecución → Cambiar tipo de entorno → GPU T4* (opcional, también funciona en CPU).

> **Dónde está cada cosa en el proyecto Pander**
>
> | Qué | Ruta (dentro de `pander_back/`) |
> |---|---|
> | Script del modelo (el que se sube en el paso 1) | `apps/candidatos/ia/ner_documentos.py` |
> | Modelo entrenado que usa la API (resultado del paso 3) | `apps/candidatos/ia/modelo/modelo.pt` |
> | Documentación de la importación con IA | `ai/documentacion_importacion_ia.ipynb` |
>
> El modelo guardado depende de la versión del script con el que se entrenó. Si cambias `ner_documentos.py`,
> vuelve a entrenar y reemplaza `modelo.pt`.

In [ ]:
!pip install -q pytorch-crf faker seqeval

## 1. Sube `ner_documentos.py`

In [ ]:
from google.colab import files
files.upload()  # selecciona ner_documentos.py

## 2. (Opcional) Sube tus datos reales anotados en JSONL

In [ ]:
# files.upload()  # descomenta y sube mis_datos.jsonl

## 3. Entrenar

In [ ]:
!python ner_documentos.py train --synth 8000 --epochs 12 --out modelo.pt
# Con datos reales:
# !python ner_documentos.py train --synth 8000 --real mis_datos.jsonl --epochs 15 --out modelo.pt

## 4. Descargar el modelo entrenado

In [ ]:
files.download('modelo.pt')

## 5. Probar con un texto

In [ ]:
!pip install -q markitdown easyocr pymupdf
# files.upload()  # sube un PDF/imagen
# !python ner_documentos.py predict --model modelo.pt tu_archivo.pdf

## 6. Poner el modelo nuevo en el proyecto

1. Copia el `modelo.pt` que descargaste en el paso 4 a `apps/candidatos/ia/modelo/modelo.pt` (reemplaza el anterior).
2. Reinicia el servidor Django (`python manage.py runserver`). El modelo se carga una sola vez por proceso,
   así que sin reiniciar sigue usando el viejo. La caché de resultados se invalida sola porque depende del
   tamaño y la fecha del archivo.
3. Comprueba que quedó bien: abre `http://127.0.0.1:8000/api/importacion/cv/estado/` y revisa que diga
   `"disponible": true`.
4. Prueba con algunas hojas de vida reales en **Importación Masiva → Candidatos → Hojas de vida (IA)** usando
   el botón *Analizar* (no guarda nada). Si falla en algún campo, agrégalo a tus datos reales anotados
   (paso 2) y vuelve a entrenar.

Más detalle del flujo completo en `ai/documentacion_importacion_ia.ipynb`.